# Ablation Study: GMF vs MLP vs NeuMF

Porównanie składowych modelu NeuMF:
- **GMF only** — tylko gałąź Generalized Matrix Factorization
- **MLP only** — tylko gałąź Multi-Layer Perceptron
- **NeuMF** — pełny model (GMF + MLP)

Oraz wpływ rozmiaru embeddingów: 8, 16, 32, 64.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data.dataset import get_dataloaders
from models.ncf import GMF, MLP, NeuMF
from evaluate import evaluate_ranking

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load data once — reuse across experiments
train_loader, val_loader, test_loader, n_users, n_items, user_positives = get_dataloaders(
    batch_size=256, n_neg=4
)
print(f'Users: {n_users}, Items: {n_items}')

## 1. Pomocnicza funkcja treningowa

In [ ]:
def quick_train(model, epochs=10):
    """Train model and return val losses per epoch."""
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()
    val_losses = []

    for epoch in range(1, epochs + 1):
        model.train()
        for users, items, labels in train_loader:
            users, items, labels = users.to(device), items.to(device), labels.to(device)
            optimizer.zero_grad()
            nn.BCELoss()(model(users, items), labels).backward()
            optimizer.step()

        model.eval()
        total = 0.0
        with torch.no_grad():
            for users, items, labels in val_loader:
                users, items, labels = users.to(device), items.to(device), labels.to(device)
                total += criterion(model(users, items), labels).item() * len(labels)
        val_losses.append(total / len(val_loader.dataset))
        print(f'  Epoch {epoch}: val_loss={val_losses[-1]:.4f}')

    return val_losses

## 2. GMF only

In [ ]:
# GMF with output head for standalone training
class GMFModel(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32):
        super().__init__()
        from models.ncf import GMF
        self.gmf = GMF(n_users, n_items, emb_dim)
        self.out = nn.Linear(emb_dim, 1)

    def forward(self, u, i):
        return torch.sigmoid(self.out(self.gmf(u, i)).squeeze(-1))

gmf_model = GMFModel(n_users, n_items).to(device)
print('Training GMF only...')
gmf_losses = quick_train(gmf_model)

## 3. MLP only

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32):
        super().__init__()
        from models.ncf import MLP
        self.mlp = MLP(n_users, n_items, emb_dim, layers=[64, 32, 16])
        self.out = nn.Linear(16, 1)

    def forward(self, u, i):
        return torch.sigmoid(self.out(self.mlp(u, i)).squeeze(-1))

mlp_model = MLPModel(n_users, n_items).to(device)
print('Training MLP only...')
mlp_losses = quick_train(mlp_model)

## 4. NeuMF (GMF + MLP)

In [ ]:
neumf_model = NeuMF(n_users, n_items, gmf_dim=32, mlp_dim=32).to(device)
print('Training NeuMF...')
neumf_losses = quick_train(neumf_model)

## 5. Porównanie krzywych val loss

In [ ]:
epochs = range(1, len(gmf_losses) + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs, gmf_losses,   label='GMF only')
plt.plot(epochs, mlp_losses,   label='MLP only')
plt.plot(epochs, neumf_losses, label='NeuMF')
plt.xlabel('Epoch'); plt.ylabel('Val BCE Loss')
plt.title('Ablation: GMF vs MLP vs NeuMF')
plt.legend(); plt.tight_layout()
plt.savefig('../results/ablation_val_loss.png', dpi=150)
plt.show()

## 6. Wpływ rozmiaru embeddingów (NeuMF)

In [ ]:
results = {}
for emb_dim in [8, 16, 32, 64]:
    print(f'\nemb_dim={emb_dim}')
    m = NeuMF(n_users, n_items, gmf_dim=emb_dim, mlp_dim=emb_dim).to(device)
    losses = quick_train(m, epochs=10)
    metrics = evaluate_ranking(m, test_loader, n_items, user_positives, device, k=10)
    results[emb_dim] = metrics

df_emb = pd.DataFrame(results).T
print(df_emb)